Дорогой студент!

В домашнем задании Lite вам предлагается поработать подробнее с параметрами словаря и формированием гиперпараметров нейронной сети. Создайте 9 нейросетей с различными гиперпараметрами (см. пунтк 2 и 3)

 Для этого необходимо:

  1. Воссоздать ноутбук, аналогичный ноутбуку практической части №1, загрузив при этом необходимую нам базу (код уже доступен в ноутбуке).

  2. Задать в ноутбуке следующие параметры для размера словаря, ширины окна и шага:

    - Размер словаря - от 10000 до 20000 (выбрать меньшее значение диапазона, если будет перегрузка ОЗУ и перезапуск подключения к Colaboratory)
    - Ширина окна - от 1000 до 2000
    - Шаг - от 100 до 500 (на обучение лучше влияет наименьший шаг, но это может перегрузить ОЗУ).

  3. Создать архитектуру сети и задать гиперпараметры. Можно воспользоваться шаблоном:
  
   - Добавьте модель прямого распространения **Sequential()**
   - Добавьте один или несколько полносвязных (**Dense**) слоёв
   - Добавьте слои **Dropout()** и **BatchNormalization()**
   - Добавьте выходной полносвязный слой с количеством нейронов, соответствующим количеству классов (число писателей)
  
   Напомним, что точность сети можно проверить по значению показателя 'val_accuracy' на конце каждой эпохи.
   

## 1. Импорт библиотек, загрузка датасета, подготовка текстов и токенизация

In [13]:
# 1. Импорт библиотек
import numpy as np
from tensorflow.keras import utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Flatten, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import gdown
import os
import re
import matplotlib.pyplot as plt

# 2. Загрузка и распаковка датасета
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)
!unzip -qo writers.zip -d writers/

# 3. Константы и загрузка текстов
FILE_DIR = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST = 'тестовая'

CLASS_LIST = []
text_train = []
text_test = []

for file_name in os.listdir(FILE_DIR):
    m = re.match('\((.+)\) (\S+)_', file_name)
    if m:
        class_name = m[1]
        subset_name = m[2].lower()
        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name
        if is_train or is_test:
            if class_name not in CLASS_LIST:
                CLASS_LIST.append(class_name)
                text_train.append('')
                text_test.append('')
            cls = CLASS_LIST.index(class_name)
            with open(f'{FILE_DIR}/{file_name}', 'r') as f:
                text = f.read()
            subset = text_train if is_train else text_test
            subset[cls] += ' ' + text.replace('\n', ' ')

CLASS_COUNT = len(CLASS_LIST)
print(f"Классы: {CLASS_LIST}")

# 4. Фиксированный размер словаря
VOCAB_SIZE = 15000

<>:27: SyntaxWarning: invalid escape sequence '\('
<>:27: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_5691/1120486570.py:27: SyntaxWarning: invalid escape sequence '\('
  m = re.match('\((.+)\) (\S+)_', file_name)


Классы: ['Клиффорд_Саймак', 'Булгаков', 'О. Генри', 'Стругацкие', 'Рэй Брэдберри', 'Макс Фрай']
Данные загружены. Готов к токенизации.


## 2. Функции разбиения последовательностей и создания модели

In [14]:
# Функция разбиения последовательности на отрезки скользящим окном
def split_sequence(sequence, win_size, hop):
    return [sequence[i:i + win_size] for i in range(0, len(sequence) - win_size + 1, hop)]

# Функция формирования выборок из последовательностей индексов
def vectorize_sequence(seq_list, win_size, hop):
    class_count = len(seq_list)
    x, y = [], []
    for cls in range(class_count):
        vectors = split_sequence(seq_list[cls], win_size, hop)
        x += vectors
        y += [utils.to_categorical(cls, class_count)] * len(vectors)
    return np.array(x), np.array(y)

# Функция создания модели с Embedding слоем (единая для всех экспериментов)
def create_model(win_size, vocab_size, class_count):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=win_size))
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.3))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.3))
    model.add(BatchNormalization())
    model.add(Dense(class_count, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## 3. Токенизация текстов и преобразование в последовательности индексов

In [ ]:
# Токенизация
tokenizer = Tokenizer(num_words=VOCAB_SIZE,
                      filters='!"#$%&()*+,-–—./…:;<=>?@[\\]^_`{|}~«»\t\n\xa0\ufeff',
                      lower=True, split=' ', oov_token='неизвестное_слово')
tokenizer.fit_on_texts(text_train)

# Преобразуем тексты в последовательности индексов
seq_train = tokenizer.texts_to_sequences(text_train)
seq_test = tokenizer.texts_to_sequences(text_test)

Токенизация завершена. Последовательности индексов готовы.


## 4. Эксперимент 1: WIN_SIZE=1000, WIN_HOP=100

In [16]:
print("\n=== ЭКСПЕРИМЕНТ 1: WIN_SIZE=1000, HOP=100 ===")

WIN_SIZE = 1000
WIN_HOP = 100

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc1 = max(history.history['val_accuracy'])
print(f"Результат 1: Лучшая val_accuracy = {acc1:.4f}")


=== ЭКСПЕРИМЕНТ 1: WIN_SIZE=1000, HOP=100 ===
Размер выборки: Train (17640, 1000), Test (6686, 1000)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 13s 51ms/step - accuracy: 0.7618 - loss: 0.6851 - val_accuracy: 0.6279 - val_loss: 1.1436
Epoch 2/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9980 - loss: 0.0212 - val_accuracy: 0.7300 - val_loss: 0.8306
Epoch 3/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9997 - loss: 0.0059 - val_accuracy: 0.7459 - val_loss: 0.7968
Epoch 4/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.9999 - loss: 0.0033 - val_accuracy: 0.7549 - val_loss: 0.7879
Epoch 5/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9999 - loss: 0.0020 - val_accuracy: 0.7550 - val_loss: 0.7968
Epoch 6/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9999 - loss: 0.0015 - val_accuracy: 0.7653 - val_loss: 0.7767
Epoch 7/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.7556 - val_loss: 0.8206
Epoch 8/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9999 - loss: 0.0010 - val_acc

## 5. Эксперимент 2: WIN_SIZE=1000, WIN_HOP=300

In [17]:
print("\n=== ЭКСПЕРИМЕНТ 2: WIN_SIZE=1000, HOP=300 ===")

WIN_SIZE = 1000
WIN_HOP = 300

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc2 = max(history.history['val_accuracy'])
print(f"Результат 2: Лучшая val_accuracy = {acc2:.4f}")


=== ЭКСПЕРИМЕНТ 2: WIN_SIZE=1000, HOP=300 ===
Размер выборки: Train (5883, 1000), Test (2231, 1000)
Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 0.4746 - loss: 1.5119 - val_accuracy: 0.3102 - val_loss: 1.6084
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9371 - loss: 0.2890 - val_accuracy: 0.4827 - val_loss: 1.2980
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9966 - loss: 0.0542 - val_accuracy: 0.5235 - val_loss: 1.2060
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9993 - loss: 0.0228 - val_accuracy: 0.5742 - val_loss: 1.1577
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9998 - loss: 0.0141 - val_accuracy: 0.5903 - val_loss: 1.1021
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9993 - loss: 0.0097 - val_accuracy: 0.6091 - val_loss: 1.0969
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9998 - loss: 0.0074 - val_accuracy: 0.6369 - val_loss: 1.0410
Epoch 8/

## 6. Эксперимент 3: WIN_SIZE=1000, WIN_HOP=500

In [18]:
print("\n=== ЭКСПЕРИМЕНТ 3: WIN_SIZE=1000, HOP=500 ===")

WIN_SIZE = 1000
WIN_HOP = 500

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc3 = max(history.history['val_accuracy'])
print(f"Результат 3: Лучшая val_accuracy = {acc3:.4f}")


=== ЭКСПЕРИМЕНТ 3: WIN_SIZE=1000, HOP=500 ===
Размер выборки: Train (3531, 1000), Test (1340, 1000)
Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 10s 157ms/step - accuracy: 0.3829 - loss: 1.7837 - val_accuracy: 0.3022 - val_loss: 1.8274
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.8318 - loss: 0.5554 - val_accuracy: 0.4045 - val_loss: 1.4542
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9626 - loss: 0.1905 - val_accuracy: 0.4037 - val_loss: 1.4104
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9926 - loss: 0.0727 - val_accuracy: 0.4246 - val_loss: 1.3734
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9975 - loss: 0.0359 - val_accuracy: 0.4037 - val_loss: 1.4156
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9983 - loss: 0.0241 - val_accuracy: 0.4127 - val_loss: 1.4163
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9997 - loss: 0.0180 - val_accuracy: 0.4440 - val_loss: 1.3850
Epoch 8/

## 7. Эксперимент 4: WIN_SIZE=1500, WIN_HOP=100

In [19]:
print("\n=== ЭКСПЕРИМЕНТ 4: WIN_SIZE=1500, HOP=100 ===")

WIN_SIZE = 1500
WIN_HOP = 100

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc4 = max(history.history['val_accuracy'])
print(f"Результат 4: Лучшая val_accuracy = {acc4:.4f}")


=== ЭКСПЕРИМЕНТ 4: WIN_SIZE=1500, HOP=100 ===
Размер выборки: Train (17610, 1500), Test (6656, 1500)
Epoch 1/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - accuracy: 0.8287 - loss: 0.5142 - val_accuracy: 0.6980 - val_loss: 0.9649
Epoch 2/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9985 - loss: 0.0148 - val_accuracy: 0.7809 - val_loss: 0.6721
Epoch 3/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9997 - loss: 0.0046 - val_accuracy: 0.7931 - val_loss: 0.6123
Epoch 4/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.9998 - loss: 0.0028 - val_accuracy: 0.7828 - val_loss: 0.6490
Epoch 5/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 0.8048 - val_loss: 0.6158
Epoch 6/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.7942 - val_loss: 0.6366
Epoch 7/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9999 - loss: 0.0011 - val_accuracy: 0.8012 - val_loss: 0

## 8. Эксперимент 5: WIN_SIZE=1500, WIN_HOP=300

In [20]:
print("\n=== ЭКСПЕРИМЕНТ 5: WIN_SIZE=1500, HOP=300 ===")

WIN_SIZE = 1500
WIN_HOP = 300

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc5 = max(history.history['val_accuracy'])
print(f"Результат 5: Лучшая val_accuracy = {acc5:.4f}")


=== ЭКСПЕРИМЕНТ 5: WIN_SIZE=1500, HOP=300 ===
Размер выборки: Train (5872, 1500), Test (2220, 1500)
Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 12s 140ms/step - accuracy: 0.5293 - loss: 1.3456 - val_accuracy: 0.3685 - val_loss: 1.5369
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9670 - loss: 0.1894 - val_accuracy: 0.5486 - val_loss: 1.2310
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9978 - loss: 0.0330 - val_accuracy: 0.6486 - val_loss: 1.0504
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9993 - loss: 0.0152 - val_accuracy: 0.6820 - val_loss: 0.9832
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9990 - loss: 0.0109 - val_accuracy: 0.6851 - val_loss: 0.9733
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9997 - loss: 0.0075 - val_accuracy: 0.7104 - val_loss: 0.8862
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.9998 - loss: 0.0056 - val_accuracy: 0.7077 - val_loss: 0.9242
Epoch 8/

## 9. Эксперимент 6: WIN_SIZE=1500, WIN_HOP=500

In [21]:
print("\n=== ЭКСПЕРИМЕНТ 6: WIN_SIZE=1500, HOP=500 ===")

WIN_SIZE = 1500
WIN_HOP = 500

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc6 = max(history.history['val_accuracy'])
print(f"Результат 6: Лучшая val_accuracy = {acc6:.4f}")


=== ЭКСПЕРИМЕНТ 6: WIN_SIZE=1500, HOP=500 ===
Размер выборки: Train (3525, 1500), Test (1334, 1500)
Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 11s 180ms/step - accuracy: 0.4113 - loss: 1.7044 - val_accuracy: 0.3388 - val_loss: 1.8138
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8769 - loss: 0.4810 - val_accuracy: 0.4198 - val_loss: 1.4606
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9818 - loss: 0.1372 - val_accuracy: 0.4723 - val_loss: 1.3525
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9952 - loss: 0.0554 - val_accuracy: 0.4783 - val_loss: 1.3440
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9983 - loss: 0.0315 - val_accuracy: 0.4865 - val_loss: 1.3422
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9983 - loss: 0.0203 - val_accuracy: 0.4985 - val_loss: 1.3518
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9989 - loss: 0.0159 - val_accuracy: 0.5075 - val_loss: 1.3161
Epoch 8/

## 10. Эксперимент 7: WIN_SIZE=2000, WIN_HOP=100

In [22]:
print("\n=== ЭКСПЕРИМЕНТ 7: WIN_SIZE=2000, HOP=100 ===")

WIN_SIZE = 2000
WIN_HOP = 100

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc7 = max(history.history['val_accuracy'])
print(f"Результат 7: Лучшая val_accuracy = {acc7:.4f}")


=== ЭКСПЕРИМЕНТ 7: WIN_SIZE=2000, HOP=100 ===
Размер выборки: Train (17580, 2000), Test (6626, 2000)
Epoch 1/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 19s 72ms/step - accuracy: 0.8449 - loss: 0.4546 - val_accuracy: 0.7025 - val_loss: 0.9682
Epoch 2/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.9990 - loss: 0.0139 - val_accuracy: 0.7791 - val_loss: 0.7183
Epoch 3/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.9998 - loss: 0.0042 - val_accuracy: 0.7999 - val_loss: 0.6863
Epoch 4/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.9998 - loss: 0.0023 - val_accuracy: 0.8006 - val_loss: 0.6677
Epoch 5/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 0.8038 - val_loss: 0.6503
Epoch 6/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.8204 - val_loss: 0.6363
Epoch 7/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 1.0000 - loss: 8.2204e-04 - val_accuracy: 0.8189 - val_los

## 11. Эксперимент 8: WIN_SIZE=2000, WIN_HOP=300

In [23]:
print("\n=== ЭКСПЕРИМЕНТ 8: WIN_SIZE=2000, HOP=300 ===")

WIN_SIZE = 2000
WIN_HOP = 300

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc8 = max(history.history['val_accuracy'])
print(f"Результат 8: Лучшая val_accuracy = {acc8:.4f}")


=== ЭКСПЕРИМЕНТ 8: WIN_SIZE=2000, HOP=300 ===
Размер выборки: Train (5861, 2000), Test (2211, 2000)
Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 14s 139ms/step - accuracy: 0.5755 - loss: 1.2168 - val_accuracy: 0.6196 - val_loss: 1.3388
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9797 - loss: 0.1388 - val_accuracy: 0.7038 - val_loss: 0.9358
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9978 - loss: 0.0293 - val_accuracy: 0.7422 - val_loss: 0.8142
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9991 - loss: 0.0135 - val_accuracy: 0.7463 - val_loss: 0.7844
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9998 - loss: 0.0092 - val_accuracy: 0.7603 - val_loss: 0.7739
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.9998 - loss: 0.0065 - val_accuracy: 0.7499 - val_loss: 0.7959
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9998 - loss: 0.0051 - val_accuracy: 0.7607 - val_loss: 0.7648
Epoch 8/

## 12. Эксперимент 9: WIN_SIZE=2000, WIN_HOP=500

In [24]:
print("\n=== ЭКСПЕРИМЕНТ 9: WIN_SIZE=2000, HOP=500 ===")

WIN_SIZE = 2000
WIN_HOP = 500

x_train, y_train = vectorize_sequence(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = vectorize_sequence(seq_test, WIN_SIZE, WIN_HOP)
print(f"Размер выборки: Train {x_train.shape}, Test {x_test.shape}")

model = create_model(WIN_SIZE, VOCAB_SIZE, CLASS_COUNT)
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=1)

acc9 = max(history.history['val_accuracy'])
print(f"Результат 9: Лучшая val_accuracy = {acc9:.4f}")


=== ЭКСПЕРИМЕНТ 9: WIN_SIZE=2000, HOP=500 ===
Размер выборки: Train (3519, 2000), Test (1328, 2000)
Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 229ms/step - accuracy: 0.4297 - loss: 1.6044 - val_accuracy: 0.3426 - val_loss: 1.6515
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9048 - loss: 0.4395 - val_accuracy: 0.4217 - val_loss: 1.4649
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9841 - loss: 0.1300 - val_accuracy: 0.4616 - val_loss: 1.3761
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9946 - loss: 0.0551 - val_accuracy: 0.5038 - val_loss: 1.2782
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9977 - loss: 0.0282 - val_accuracy: 0.5226 - val_loss: 1.2497
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9989 - loss: 0.0202 - val_accuracy: 0.5459 - val_loss: 1.1803
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9991 - loss: 0.0134 - val_accuracy: 0.5633 - val_loss: 1.1885
Epoch 8/

## 13. Сравнение результатов всех экспериментов

In [26]:
print("Эксперимент | WIN_SIZE | WIN_HOP | Лучшая val_accuracy")
print("------------------------------------------------------")
print(f"     1      |   1000   |   100   |      {acc1:.4f}")
print(f"     2      |   1000   |   300   |      {acc2:.4f}")
print(f"     3      |   1000   |   500   |      {acc3:.4f}")
print(f"     4      |   1500   |   100   |      {acc4:.4f}")
print(f"     5      |   1500   |   300   |      {acc5:.4f}")
print(f"     6      |   1500   |   500   |      {acc6:.4f}")
print(f"     7      |   2000   |   100   |      {acc7:.4f}")
print(f"     8      |   2000   |   300   |      {acc8:.4f}")
print(f"     9      |   2000   |   500   |      {acc9:.4f}")

best_acc = max([acc1, acc2, acc3, acc4, acc5, acc6, acc7, acc8, acc9])
print(f"\nЛучший результат: {best_acc:.4f} ({best_acc*100:.2f}%)")

# Находим лучшие параметры
if best_acc == acc1: print("Лучшие параметры: WIN_SIZE=1000, WIN_HOP=100")
elif best_acc == acc2: print("Лучшие параметры: WIN_SIZE=1000, WIN_HOP=300")
elif best_acc == acc3: print("Лучшие параметры: WIN_SIZE=1000, WIN_HOP=500")
elif best_acc == acc4: print("Лучшие параметры: WIN_SIZE=1500, WIN_HOP=100")
elif best_acc == acc5: print("Лучшие параметры: WIN_SIZE=1500, WIN_HOP=300")
elif best_acc == acc6: print("Лучшие параметры: WIN_SIZE=1500, WIN_HOP=500")
elif best_acc == acc7: print("Лучшие параметры: WIN_SIZE=2000, WIN_HOP=100")
elif best_acc == acc8: print("Лучшие параметры: WIN_SIZE=2000, WIN_HOP=300")
elif best_acc == acc9: print("Лучшие параметры: WIN_SIZE=2000, WIN_HOP=500")

Эксперимент | WIN_SIZE | WIN_HOP | Лучшая val_accuracy
------------------------------------------------------
     1      |   1000   |   100   |      0.7975
     2      |   1000   |   300   |      0.6602
     3      |   1000   |   500   |      0.4649
     4      |   1500   |   100   |      0.8131
     5      |   1500   |   300   |      0.7306
     6      |   1500   |   500   |      0.5435
     7      |   2000   |   100   |      0.8248
     8      |   2000   |   300   |      0.7607
     9      |   2000   |   500   |      0.5791

Лучший результат: 0.8248 (82.48%)
Лучшие параметры: WIN_SIZE=2000, WIN_HOP=100
